In [1]:
import numpy as np
import pandas as pd
from plotly.io import show

from skfolio import Population, RiskMeasure
from skfolio.datasets import load_factors_dataset, load_sp500_dataset
from skfolio.distribution import VineCopula
from skfolio.measures import (
    cvar,
    kurtosis,
    mean,
    skew,
    standard_deviation,
    value_at_risk,
)
from skfolio.optimization import HierarchicalRiskParity, RiskBudgeting
from skfolio.preprocessing import prices_to_returns
from skfolio.prior import EntropyPooling, FactorModel, OpinionPooling, SyntheticData
from skfolio.utils.figure import plot_kde_distributions

# Load stock price and factor data
prices = load_sp500_dataset()
prices = prices[["AMD", "BAC", "GE", "JNJ", "JPM", "LLY", "PG"]]
factor_prices = load_factors_dataset()

# Convert to daily returns
X, factors = prices_to_returns(prices, factor_prices)

print("Shapes:")
print(f"X: {X.shape}")
print(f"factors: {factors.shape}")

print(X.tail())
print(factors.tail())

Shapes:
X: (2263, 7)
factors: (2263, 5)
                 AMD       BAC        GE       JNJ       JPM       LLY  \
Date                                                                     
2022-12-21  0.040430  0.015223  0.033001  0.011444  0.011248  0.023275   
2022-12-22 -0.056442 -0.008848 -0.014582 -0.003655 -0.011355 -0.007339   
2022-12-23  0.010335  0.002443  0.000235  0.002539  0.004749  0.007090   
2022-12-27 -0.019374  0.001875  0.012849 -0.000280  0.003504 -0.008208   
2022-12-28 -0.011064  0.007360 -0.010502 -0.004341  0.005463  0.000932   

                  PG  
Date                  
2022-12-21  0.009170  
2022-12-22  0.002308  
2022-12-23  0.002825  
2022-12-27  0.008713  
2022-12-28 -0.012926  
                MTUM      QUAL      SIZE      USMV      VLUE
Date                                                        
2022-12-21  0.014312  0.017884  0.014371  0.012005  0.013246
2022-12-22 -0.010977 -0.015411 -0.012070 -0.007315 -0.011989
2022-12-23  0.010897  0.005889  0.00

In [2]:
def summary(X: pd.DataFrame, sample_weight: np.ndarray | None = None) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "Mean": mean(X, sample_weight=sample_weight),
            "Volatility": standard_deviation(X, sample_weight=sample_weight),
            "Skew": skew(X, sample_weight=sample_weight),
            "Kurtosis": kurtosis(X, sample_weight=sample_weight),
            "VaR at 95%": value_at_risk(X, beta=0.95, sample_weight=sample_weight),
            "CVaR at 95%": cvar(X, beta=0.95, sample_weight=sample_weight),
        }
    )


summary(X)

,Mean,Volatility,Skew,Kurtosis,VaR at 95%,CVaR at 95%
AMD,0.001902,0.037314,1.323839,22.604470,0.052970,0.078580
BAC,0.000581,0.019822,0.283198,13.312827,0.028821,0.044680
GE,-0.000099,0.021970,0.179690,9.851668,0.032739,0.051328
JNJ,0.000466,0.011443,-0.261929,12.610478,0.016330,0.027007
JPM,0.000621,0.017354,0.343518,17.000055,0.025351,0.038524
LLY,0.001102,0.016708,0.910982,14.984163,0.022783,0.034948
PG,0.000465,0.011699,0.261102,16.276273,0.016019,0.027307


In [3]:
opinion_1 = EntropyPooling(cvar_views=["AMD == 0.10"])

opinion_2 = EntropyPooling(
    mean_views=["AMD >= BAC", "JPM <= prior(JPM) * 0.8"],
    cvar_views=["GE == 0.12"],
)

opinion_pooling = OpinionPooling(
    estimators=[("opinion_1", opinion_1), ("opinion_2", opinion_2)],
    opinion_probabilities=[0.4, 0.5],
)

opinion_pooling.fit(X)

sample_weight = opinion_pooling.return_distribution_.sample_weight
summary(X, sample_weight=sample_weight)

,Mean,Volatility,Skew,Kurtosis,VaR at 95%,CVaR at 95%
AMD,-0.000760,0.041141,0.456986,17.707707,0.062590,0.104290
BAC,-0.001752,0.026380,-2.124740,17.464876,0.034702,0.078480
GE,-0.002815,0.028597,-1.692467,12.656108,0.040912,0.089127
JNJ,-0.000321,0.012897,-0.908678,11.038584,0.019654,0.036289
JPM,-0.001571,0.024030,-2.501122,20.897790,0.029561,0.071314
LLY,0.000002,0.018725,-0.131088,13.343008,0.025866,0.047965
PG,-0.000366,0.013508,-0.847842,15.226395,0.018663,0.037963


In [4]:
plot_kde_distributions(
    X,
    sample_weight=sample_weight,
    percentile_cutoff=0.05,
    title="Distribution of Asset Returns (Prior vs. Posterior)",
    unweighted_suffix="Prior",
    weighted_suffix="Posterior",
)

In [5]:
model = RiskBudgeting(
    risk_measure=RiskMeasure.CVAR, cvar_beta=0.9, prior_estimator=opinion_pooling
)

model.fit(X)

print(model.weights_)

[0.08085591 0.09789838 0.09720951 0.21843335 0.10682473 0.1722557
 0.22652242]


In [6]:
factor_opinion_1 = EntropyPooling(
    mean_views=["QUAL == -0.0005"], cvar_views=["SIZE == 0.08"]
)
factor_opinion_2 = EntropyPooling(cvar_views=["SIZE == 0.09"])

factor_opinion_pooling = OpinionPooling(
    estimators=[("opinion_1", factor_opinion_1), ("opinion_2", factor_opinion_2)],
    opinion_probabilities=[0.6, 0.4],
)

factor_model = FactorModel(factor_prior_estimator=factor_opinion_pooling)

model = RiskBudgeting(risk_measure=RiskMeasure.CVAR, prior_estimator=factor_model)


model.fit(X, factors)
print(model.weights_)

sample_weight = model.prior_estimator_.return_distribution_.sample_weight
summary(factors, sample_weight)

[0.09333241 0.09726867 0.1092549  0.21357225 0.10861785 0.1764545
 0.20149942]


,Mean,Volatility,Skew,Kurtosis,VaR at 95%,CVaR at 95%
MTUM,-0.001654,0.022502,-3.032968,18.744647,0.027900,0.077008
QUAL,-0.001331,0.019848,-2.621010,16.934030,0.025156,0.068022
SIZE,-0.002226,0.023698,-3.627090,21.842691,0.025651,0.084100
USMV,-0.001495,0.018384,-3.306216,20.498260,0.020091,0.064411
VLUE,-0.002084,0.023367,-3.237838,19.459370,0.026067,0.081470


In [7]:
vine = VineCopula(log_transform=True, n_jobs=-1, random_state=0)

factor_synth = SyntheticData(n_samples=100_000, distribution_estimator=vine)

factor_opinion_1 = EntropyPooling(cvar_views=["SIZE == 0.15"])
factor_opinion_2 = EntropyPooling(cvar_views=["SIZE == 0.20"])

factor_opinion_pooling = OpinionPooling(
    prior_estimator=factor_synth,
    estimators=[("opinion_1", factor_opinion_1), ("opinion_2", factor_opinion_2)],
    opinion_probabilities=[0.6, 0.4],
)

factor_model = FactorModel(factor_prior_estimator=factor_opinion_pooling)

model = HierarchicalRiskParity(
    risk_measure=RiskMeasure.CVAR, prior_estimator=factor_model
)

model.fit(X, factors)
print(model.weights_)

[0.04468221 0.08506756 0.09396145 0.17199215 0.09416958 0.13640712
 0.37371993]


In [8]:
fitted_vine = model.prior_estimator_.factor_prior_estimator_.prior_estimator_.distribution_estimator_

In [9]:
model = HierarchicalRiskParity(risk_measure=RiskMeasure.CVAR)

model.fit(X)
print(model.weights_)

portfolio = model.predict(X)
portfolio.name = "HRP Unstressed"

# Add to a Population for better comparison with the stressed portfolios.
population = Population([portfolio])

[0.06021919 0.1059091  0.08763318 0.18841288 0.11675892 0.25472467
 0.18634206]


In [10]:
vine = VineCopula(log_transform=True, n_jobs=-1, random_state=0)

synth = SyntheticData(n_samples=100_000, distribution_estimator=vine)

opinion_1 = EntropyPooling(cvar_beta=0.90, cvar_views=["AMD == 0.08"])
opinion_2 = EntropyPooling(cvar_views=["AMD == 0.10"])

opinion_pooling = OpinionPooling(
    prior_estimator=synth,
    estimators=[("opinion_1", opinion_1), ("opinion_2", opinion_2)],
    opinion_probabilities=[0.6, 0.4],
)

opinion_pooling.fit(X)

# We retrieve the stressed distribution:
stressed_dist = opinion_pooling.return_distribution_

# We stress-test our portfolio:
stressed_ptf = model.predict(stressed_dist)

# Add the stressed portfolio to the population
stressed_ptf.name = "HRP Stressed"
population.append(stressed_ptf)

In [11]:
factor_synth = SyntheticData(n_samples=100_000, distribution_estimator=vine)

factor_opinion_1 = EntropyPooling(cvar_beta=0.90, cvar_views=["QUAL == 0.10"])
factor_opinion_2 = EntropyPooling(cvar_views=["QUAL == 0.12"])

factor_opinion_pooling = OpinionPooling(
    prior_estimator=factor_synth,
    estimators=[("opinion_1", factor_opinion_1), ("opinion_2", factor_opinion_2)],
    opinion_probabilities=[0.6, 0.4],
)

factor_model = FactorModel(factor_prior_estimator=factor_opinion_pooling)

factor_model.fit(X, factors)

# We retrieve the stressed distribution:
stressed_dist = factor_model.return_distribution_

# We stress-test our portfolio:
stressed_ptf = model.predict(stressed_dist)

# Add the stressed portfolio to the population
stressed_ptf.name = "HRP Factor Stressed"
population.append(stressed_ptf)

In [12]:
pop_summary = population.summary()
pop_summary.loc[
    [
        "Mean",
        "Standard Deviation",
        "CVaR at 95%",
        "Annualized Sharpe Ratio",
        "Worst Realization",
    ]
]

,HRP Unstressed,HRP Stressed,HRP Factor Stressed
Mean,0.070%,0.034%,-0.45%
Standard Deviation,1.14%,1.24%,2.78%
CVaR at 95%,2.64%,2.94%,10.38%
Annualized Sharpe Ratio,0.96,0.44,-2.59
Worst Realization,9.16%,23.95%,17.16%


In [13]:
fig = population.plot_returns_distribution(percentile_cutoff=0.05)
show(fig)